# 08. Térbeli Autokorreláció és Moran-I Teszt

**Cél:** a térbeli függőség matematikai bizonyítása és a klasszikus OLS-modell torzításának feltárása.

---

### 📖 Tobler első földrajzi törvénye (1970)
*„Minden dolog összefügg minden más dologgal, de a közeli dolgok jobban összefüggnek, mint a távoliak."* Az ingatlanpiacon ez spillover hatást jelent: egy drága lakás mellett nagyobb eséllyel áll egy másik drága lakás. Ha ez fennáll, sérül az OLS hibatag-függetlenségi feltétele.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local
from statsmodels.stats.multitest import multipletests

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
df_raw = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].dropna(subset=['geokodolt_lat', 'geokodolt_lon', 'nm_ar_huf']).copy()

# Térbeli aggregáció: Épület szinten összevont pontok a szomszédsági topológiához
df_geo = df_raw.groupby(['geokodolt_lon', 'geokodolt_lat']).agg(
    nm_ar_huf=('nm_ar_huf', 'mean'),
    alapterulet_nm=('alapterulet_nm', 'mean'),
    allapot_kod=('allapot_kod', 'mean'),
    szobaszam_osszes=('szobaszam_osszes', 'mean'),
    cim_teljes=('cim_teljes', 'first'),
    varosresz=('varosresz', 'first')
).reset_index()

df_geo['allapot_kod'] = df_geo['allapot_kod'].fillna(df_geo['allapot_kod'].median())
coords = np.column_stack((df_geo['geokodolt_lon'], df_geo['geokodolt_lat']))

# Lokális síkvetület (WGS84 fok -> méter): a KNN szomszédságot NEM fokban, hanem
# torzításmentes méter-távolságon számoljuk (x = lon*cos(mid_lat)*111320, y = lat*110540).
mid_lat = np.median(df_geo['geokodolt_lat'])
coords_proj = np.column_stack((
    df_geo['geokodolt_lon'].values * np.cos(np.radians(mid_lat)) * 111320.0,
    df_geo['geokodolt_lat'].values * 110540.0
))

# Robusztusság-ellenőrzéshez: log-négyzetméterár (a fő elemzés a nyers nm_ar_huf-on marad)
df_geo['log_nm_ar'] = np.log(df_geo['nm_ar_huf'])

print(f"Eredeti pontos eladó minta: {len(df_raw)} db.")
print(f"Épület szinten aggregált térbeli objektumok: {len(df_geo)} db.")

### 1. Súlyozott Térbeli Hálózat és a Globális Moran-I

**📌 Módszertan:**
Először K-legközelebbi szomszéd (KNN) súlymátrixot építünk — nem WGS84-fokban, hanem lokális síkvetületi (méter) koordinátákon —, amely definiálja a szomszédságot, majd a Globális Moran-I statisztikát számítjuk a négyzetméterárra (robusztusság-ellenőrzésként a log-árra is).

**🔍 Értelmezés:**
Ha a Moran-I szignifikánsan pozitív, az matematikailag igazolja, hogy az OLS nem kezeli a térbeli struktúrát — térökonometriai (SAR/SEM) modellekre van szükség.

In [ ]:
# KNN súlymátrix (k=8) lokális síkvetületi (méter) koordinátákon és Globális Moran's I becslés
w_knn = KNN.from_array(coords_proj, k=8)
w_knn.transform = 'R'

y_val = df_geo['nm_ar_huf'].values
np.random.seed(42)
moran_global = Moran(y_val, w_knn, permutations=999)

# Robusztusság-ellenőrzés: Moran-I a log-négyzetméteráron is (a fő eredmény a nyers áron marad)
y_log = df_geo['log_nm_ar'].values
np.random.seed(42)
moran_global_log = Moran(y_log, w_knn, permutations=999)

# KPI-szövegek a TÉNYLEGES I-értékből és permutációs p-ből (nem hardkódolt címkék)
I_val = float(moran_global.I)
p_val = float(moran_global.p_sim)
if I_val >= 0.3 and p_val < 0.05:
    pattern_txt = 'Erős Pozitív Klaszter'
elif I_val >= 0.1:
    pattern_txt = 'Gyenge-Közepes Pozitív Klaszter' if p_val < 0.05 else 'Gyenge Pozitív Autokorreláció (nem szignifikáns)'
elif I_val > 0:
    pattern_txt = 'Gyenge Pozitív Autokorreláció'
else:
    pattern_txt = 'Nem Pozitív Térbeli Autokorreláció'
p_desc = 'Statisztikailag szignifikáns (p < 0.05)' if p_val < 0.05 else 'Nem szignifikáns (p >= 0.05)'

kpi_cards = [
    ("Globális Moran's I", f"{I_val:.3f}", "Térbeli autokorreláció (nm_ar_huf)", "#1e3a8a"),
    ("Várható I (Véletlen)", f"{moran_global.EI:.3f}", "H0 hipotézis értéke", "#64748b"),
    ("Z-statisztika", f"{moran_global.z_sim:.2f}", "Szignifikancia mértéke", "#059669"),
    ("Permutációs P-érték", f"{p_val:.4f}", p_desc, "#2563eb"),
    ("Térbeli Mintázat", pattern_txt, f"Moran's I = {I_val:.3f}, p = {p_val:.4f}", "#7c3aed"),
    ("Szomszédok száma (k)", "k = 8", "KNN topológia (síkvetület, méter)", "#d97706"),
    ("Moran's I (log_nm_ar)", f"{moran_global_log.I:.3f}", f"Robusztusság log-áron (p = {moran_global_log.p_sim:.4f})", "#0f766e")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Moran Pontdiagram (Scatter Plot)

**📌 Hogyan értelmezzük a 4 kvadránst?**
- **High-High (HH)**: Magas ár magas szomszédokkal $\rightarrow$ Hotspot.
- **Low-Low (LL)**: Alacsony ár alacsony szomszédokkal $\rightarrow$ Coldspot.
- **High-Low és Low-High**: Térbeli kiugró értékek (outlierek).

In [ ]:
# Interaktív Moran Pontdiagram (Plotly updatemenus dropdown 5 változóra)
vars_info = [
    ('nm_ar_huf', 'Négyzetméterár', '#2563eb'),
    ('log_nm_ar', 'Log(Négyzetméterár)', '#0891b2'),
    ('alapterulet_nm', 'Alapterület (m²)', '#059669'),
    ('szobaszam_osszes', 'Összes Szobaszám', '#7c3aed'),
    ('allapot_kod', 'Műszaki Állapot Kód', '#d97706')
]

fig1 = go.Figure()
buttons_scatter = []

for i, (col, label, colr) in enumerate(vars_info):
    y = df_geo[col].values
    z = (y - y.mean()) / y.std()
    lag_z = w_knn.sparse.dot(z)
    np.random.seed(42)
    m = Moran(y, w_knn, permutations=999)
    
    # Pontok
    fig1.add_trace(go.Scatter(
        x=z, y=lag_z, mode='markers',
        marker=dict(color=colr, size=8, opacity=0.75),
        name=label,
        visible=(i == 0),
        hovertemplate="<b>%{text}</b><br>Standardizált érték: %{x:.2f}<br>Térbeli lag: %{y:.2f}<extra></extra>",
        text=df_geo['varosresz'] + ' - ' + df_geo['cim_teljes']
    ))
    # Regressziós vonal
    x_line = np.array([z.min(), z.max()])
    y_line = m.I * x_line
    fig1.add_trace(go.Scatter(
        x=x_line, y=y_line, mode='lines',
        line=dict(color='#dc2626', width=2.5, dash='dash'),
        name=f"Regressziós meredekség (Moran I = {m.I:.3f})",
        visible=(i == 0)
    ))

for i, (col, label, colr) in enumerate(vars_info):
    vis = [False] * (len(vars_info) * 2)
    vis[i*2] = True
    vis[i*2 + 1] = True
    buttons_scatter.append(dict(
        label=label,
        method="update",
        args=[{"visible": vis},
              {"title": f"Moran Scatter Plot: {label} (k=8 KNN súlyozás)"}]
    ))

fig1.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons_scatter,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.0, xanchor="left",
        y=1.22, yanchor="top"
    )],
    title="Moran Scatter Plot (Válassz változót a legördülő menüből)",
    xaxis_title="Standardizált Érték (z)",
    yaxis_title="Térbeli Késleltetés (Spatial Lag Wz)",
    template=PLOTLY_TEMPLATE,
    height=540
)
fig1.add_hline(y=0, line_dash="dot", line_color="#94a3b8")
fig1.add_vline(x=0, line_dash="dot", line_color="#94a3b8")
fig1.show()

### 3. LISA Klaszter Térkép (Hotspotok és Coldspotok)

**📌 Mit látunk a térképen?**
- **Piros (Hotspot)**: magas érték magas szomszédokkal (High-High).
- **Kék (Coldspot)**: alacsony érték alacsony szomszédokkal (Low-Low).
- A lokális p-értékeket **FDR-korrekcióval** (Benjamini–Hochberg) értékeljük a többszörös tesztelés torzításának kivédésére.

In [ ]:
# Interaktív LISA Klaszter Térkép (Plotly updatemenus rétegváltóval)
fig_lisa = go.Figure()
buttons_lisa = []

cluster_colors = {
    'High-High (Hotspot)': '#dc2626',
    'Low-Low (Coldspot)': '#2563eb',
    'High-Low (Outlier)': '#f97316',
    'Low-High (Outlier)': '#06b6d4',
    'Nem szignifikáns': '#cbd5e1'
}

for i, (col, label, colr) in enumerate(vars_info):
    y = df_geo[col].values
    np.random.seed(42)
    lm = Moran_Local(y, w_knn, permutations=999)
    p_adj = multipletests(lm.p_sim, method='fdr_bh')[1]
    
    cats = []
    for sig, q in zip(p_adj, lm.q):  # FDR-korrekció (Benjamini-Hochberg)
        if sig < 0.05:
            if q == 1: cats.append('High-High (Hotspot)')
            elif q == 2: cats.append('Low-High (Outlier)')
            elif q == 3: cats.append('Low-Low (Coldspot)')
            elif q == 4: cats.append('High-Low (Outlier)')
        else:
            cats.append('Nem szignifikáns')
            
    df_temp = df_geo.copy()
    df_temp['cluster'] = cats
    
    for c_name, c_colr in cluster_colors.items():
        sub = df_temp[df_temp['cluster'] == c_name]
        fig_lisa.add_trace(go.Scatter(
            x=sub['geokodolt_lon'],
            y=sub['geokodolt_lat'],
            mode='markers',
            marker=dict(size=9, color=c_colr, opacity=0.85),
            name=f"{c_name} ({len(sub)} db)",
            legendgroup=col,
            visible=(i == 0),
            hovertemplate=f"<b>%{{text}}</b><br>Klaszter: {c_name}<br>Érték: %{{customdata:.1f}}<extra></extra>",
            text=sub['varosresz'] + ' - ' + sub['cim_teljes'],
            customdata=sub[col]
        ))

num_groups = len(cluster_colors)
for i, (col, label, colr) in enumerate(vars_info):
    vis = [False] * (len(vars_info) * num_groups)
    for g in range(num_groups):
        vis[i * num_groups + g] = True
    buttons_lisa.append(dict(
        label=label,
        method="update",
        args=[{"visible": vis},
              {"title": f"LISA Klaszter Térkép (Hotspotok & Coldspotok): {label}"}]
    ))

fig_lisa.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons_lisa,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.0, xanchor="left",
        y=1.18, yanchor="top"
    )],
    title="LISA Klaszter Térkép (Válassz változót a legördülő menüből)",
    xaxis_title="Földrajzi Hosszúság (Lon)",
    yaxis_title="Földrajzi Szélesség (Lat)",
    template=PLOTLY_TEMPLATE,
    height=600
)
fig_lisa.show()

### 4. Szignifikancia és Érzékenységvizsgálat

**📌 Mennyire stabilak az eredmények?**
A k szomszédszám változtatása mellett a forró- és hidegpontok magja stabilan megmarad.

In [ ]:
import json

# Többváltozós Térbeli Autokorrelációs Összehasonlítás és Kliensoldali Interaktív Widget
# A moran_data_js-t ÉLŐBEN számoljuk (nem hardkódoljuk): minden változóra és k-ra
# seedelt permutációs Moran-I fut a síkvetületi KNN súlymátrixokon.
k_list = [4, 6, 8, 10, 12, 16]
moran_data_js = {}
for col, label, colr in vars_info:
    y = df_geo[col].values
    entry = {}
    for k in k_list:
        wk = KNN.from_array(coords_proj, k=k)
        wk.transform = 'R'
        np.random.seed(42)
        m = Moran(y, wk, permutations=999)
        entry[str(k)] = {"I": round(float(m.I), 3), "z": round(float(m.z_sim), 2), "p": round(float(m.p_sim), 3)}
    moran_data_js[col] = entry

# Élő érzékenységi táblázat a ténylegesen kiszámolt eredményekből
sens_rows = []
for col, label, colr in vars_info:
    row = {"Változó": label}
    for k in k_list:
        e = moran_data_js[col][str(k)]
        row[f"k={k}"] = f"{e['I']:.3f} (p={e['p']:.3f})"
    sens_rows.append(row)
display(HTML("<b>Moran-I érzékenységvizsgálat (k szomszédszám; élőben, seedelt permutációkkal számolva):</b><br>" +
             pd.DataFrame(sens_rows).to_html(classes='table table-bordered table-striped', index=False)))

# A JS widget opciói és a beágyazott adatok is az élő eredményekből képződnek
labels_js = json.dumps({col: label for col, label, colr in vars_info})
moran_js = json.dumps(moran_data_js)

options_html = ""
for i, (col, label, colr) in enumerate(vars_info):
    e8 = moran_data_js[col]["8"]
    sel = " selected" if i == 0 else ""
    options_html += f'<option value="{col}"{sel}>{label} — k=8: I = {e8["I"]:.3f}, p = {e8["p"]:.3f}</option>\n'

html_moran_app = f'''
<div id="moran_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">Térbeli Autokorrelációs (Moran's I) Interaktív Vezérlőpult</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Élőben számolt szomszédsági (KNN k) és változó-érzékenységi eredmények böngészőben</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Widget</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap:18px; margin-bottom:18px;">
    <!-- Változó választó -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <label style="display:block; font-size:13px; font-weight:700; color:#334155; margin-bottom:8px; text-transform:uppercase;">1. Elemzett Ingatlan Változó:</label>
      <select id="m_var_sel" style="width:100%; padding:9px 12px; border-radius:6px; border:1px solid #cbd5e1; font-size:13px; background:#fff;" onchange="recalcMoran()">
{options_html}      </select>
    </div>

    <!-- Szomszédsági k csúszka -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:700; color:#334155; margin-bottom:8px; text-transform:uppercase;">
        <span>2. Szomszédok Száma (KNN k):</span> <span id="m_lbl_k" style="color:#2563eb; font-weight:800;">k = 8</span>
      </div>
      <input type="range" id="m_slider_k" min="0" max="5" value="2" step="1" style="width:100%; accent-color:#2563eb;" oninput="recalcMoran()">
      <div style="display:flex; justify-content:space-between; font-size:11px; color:#64748b; margin-top:4px;">
        <span>k=4</span><span>k=6</span><span>k=8</span><span>k=10</span><span>k=12</span><span>k=16</span>
      </div>
    </div>
  </div>

  <!-- KPI Rács -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap:14px; margin-bottom:14px;">
    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Globális Moran's I</div>
      <div id="m_res_i" style="font-size:26px; font-weight:800; color:#1d4ed8; margin:4px 0;">--</div>
      <div style="font-size:11px; color:#2563eb;">Térbeli autokorreláció</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Z-Statisztika</div>
      <div id="m_res_z" style="font-size:26px; font-weight:800; color:#15803d; margin:4px 0;">--</div>
      <div style="font-size:11px; color:#16a34a;">Szórási egység (z-score)</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">P-érték (999 perm.)</div>
      <div id="m_res_p" style="font-size:24px; font-weight:800; color:#7e22ce; margin:4px 0;">--</div>
      <div id="m_res_sig" style="font-size:11px; color:#9333ea;">Szignifikancia szint</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Térbeli Struktúra</div>
      <div id="m_res_str" style="font-size:16px; font-weight:800; color:#b45309; margin:8px 0;">--</div>
      <div style="font-size:11px; color:#d97706;">Ökonometriai következmény</div>
    </div>
  </div>

  <div id="m_res_desc" style="background:#f8fafc; border-left:4px solid #2563eb; padding:12px 16px; border-radius:6px; font-size:13px; color:#334155; line-height:1.5;">
    --
  </div>
</div>

<script>
(function() {{
  const M_DATA = {moran_js};
  const M_LABELS = {labels_js};
  const k_vals = [4, 6, 8, 10, 12, 16];

  function update() {{
    const varKey = document.getElementById('m_var_sel').value;
    const varLabel = M_LABELS[varKey] || varKey;
    const kIdx = parseInt(document.getElementById('m_slider_k').value);
    const kVal = k_vals[kIdx] || 8;

    document.getElementById('m_lbl_k').innerText = 'k = ' + kVal;

    const info = (M_DATA[varKey] && M_DATA[varKey][kVal.toString()]) || null;
    if (!info) {{ return; }}

    document.getElementById('m_res_i').innerText = info.I.toFixed(3);
    document.getElementById('m_res_z').innerText = info.z.toFixed(2);
    document.getElementById('m_res_p').innerText = info.p < 0.001 ? '< 0.001' : info.p.toFixed(3);

    let sigText = 'Nem szignifikáns (p >= 0.05)';
    let strText = 'Véletlenszerű eloszlás';
    let descText = '';
    const pTxt = info.p < 0.001 ? '< 0.001' : info.p;

    if (info.p < 0.05) {{
      sigText = info.p < 0.01 ? 'Erősen szignifikáns (p < 0.01)' : 'Szignifikáns (p < 0.05)';
      strText = info.I > 0 ? 'Pozitív Térbeli Klaszterezettség' : 'Negatív Térbeli Autokorreláció';
      descText = '<b>' + varLabel + ':</b> szignifikáns ' + (info.I > 0 ? 'pozitív' : 'negatív') + ' térbeli autokorreláció (Moran I = ' + info.I.toFixed(3) + ', p = ' + pTxt + '). A hasonló értékű ingatlanok földrajzilag tömörülnek, ami térbeli függőséget jelez: ez torzíthatja a klasszikus OLS becslést, és indokolja a 09-es SAR/SEM térökonometriai modellek használatát.';
    }} else {{
      descText = '<b>' + varLabel + ':</b> nincs szignifikáns térbeli autokorreláció (Moran I = ' + info.I.toFixed(3) + ', p = ' + pTxt + '). Az értékek földrajzilag közel véletlenszerűen szóródnak.';
    }}

    document.getElementById('m_res_sig').innerText = sigText;
    document.getElementById('m_res_str').innerText = strText;
    document.getElementById('m_res_desc').innerHTML = descText;
  }}

  window.recalcMoran = update;
  setTimeout(update, 50);
}})();
</script>
'''
display(HTML(html_moran_app))
